# myCobot 280 JN inspection: Colab side (heavy computation)

**Split of work**
| Jetson Nano 4 GB (robot) | Colab (this notebook) |
|---|---|
| moves the arm, takes images | fits the move-time model from robot logs |
| grounding calibration, cost model build | trains the RL policy (several seeds) |
| collects survey images of the part | checks PyTorch = NumPy parity |
| runs inspections with the saved NumPy policy | evaluates on held-out surveys, analyses robot logs, figures |

**Where the RL happens:** in the twin built from *your robot's data*: grounded survey features
(edge density, gradients per cell) + the robot's measured joint-time costs. The policy never sees raw
pixels and is never trained on the live arm. It is evaluated on the arm afterwards.

**Inputs:** `mycobot_framework.zip` and a `to_colab_*.zip` made on the Jetson with
`python3 sync.py pack-for-colab`. **Output:** `to_jetson_*.zip` for `python3 sync.py unpack` on the Jetson.

In [ ]:
# 1. Upload the framework zip and the to_colab zip (or copy them from Google Drive)
from google.colab import files
up = files.upload()

In [ ]:
import glob, os, subprocess
fw = [f for f in up if f.startswith('mycobot_framework')][0]
!unzip -oq "$fw" -d /content
%cd /content/mycobot_framework
for z in sorted(glob.glob('/content/to_colab_*.zip')):
    print('unpacking robot data', z)
    !unzip -oq "$z" -d /content/mycobot_framework
!cat MANIFEST.json 2>/dev/null | head -20

In [ ]:
# 2. Environment check + PyTorch/NumPy parity of the policy (must be < 1e-4)
import torch, numpy as np
print('torch', torch.__version__, '| numpy', np.__version__)
from geo_attn import parity_test, GeoAttn, n_params
print('parameters:', n_params(GeoAttn()))
print('max |torch - numpy| logit difference:', parity_test())

In [ ]:
# 3. What did the robot send?
import json, numpy as np, os, glob
import cost_model as CM
print('locked pose  :', os.path.exists('deploy/locked_pose.json'))
print('grounding    :', json.load(open('deploy/grounding.json'))['A'] if os.path.exists('deploy/grounding.json') else 'MISSING')
for ik in ('hold', 'position'):
    p = CM.path_for(ik)
    if os.path.exists(p):
        M = CM.load(ik); print(f'cost model {ik:8s}: reachable {int(M["reach"].sum())}/25, t0={float(M["t0"]):.2f}s, omega={float(M["omega"]):.1f}deg/s')
for f in ('data/surveys.npz', 'data/surveys_test.npz'):
    print(f, np.load(f)['static'].shape[0] if os.path.exists(f) else 'MISSING (training will use SYNTHETIC maps)')
print('inspection logs:', len(glob.glob('hw_logs/inspection/*/log.csv')))
if os.path.exists('hw_logs/jetson_check.json'):
    j = json.load(open('hw_logs/jetson_check.json')); print('Jetson:', j.get('board'), j.get('ram_total_gb'), 'GB; decision', j.get('decision_latency_ms', {}).get('mean'), 'ms')

In [ ]:
# 4. Fit the move-time model from measured robot moves (needs inspection logs from the robot)
import glob
for ik in ('hold', 'position'):
    logs = glob.glob(f'hw_logs/inspection/*_{ik}_*/log.csv')
    if logs:
        !python cost_model.py fit {' '.join(logs)} --ik {ik}
    else:
        print(f'no {ik} logs yet: keeping the cost model from the robot (default timing)')

## 5. Train the policy (3 seeds per task)
Report mean ± SD over seeds, never only the best seed. On a Colab CPU each run takes minutes;
a GPU is not needed for this 17.7k-parameter model.

In [ ]:
# Transformer policy (ours) AND the no-attention ablation, 3 seeds each, both tasks
TASKS, ARCHS, SEEDS, ITERS, IK = ['coverage', 'budget'], ['attn', 'mlp'], [0, 1, 2], 3000, 'hold'
for task in TASKS:
    for arch in ARCHS:
        for s in SEEDS:
            !python train.py --task {task} --ik {IK} --arch {arch} --seed {s} --iters {ITERS} --policy deploy/policy_{task}_{arch}_s{s}.npz --out hw_logs/training/{task}_{IK}_{arch}_s{s}

In [ ]:
# 6. Results across seeds (held-out surveys): transformer vs ablation vs classical planners.
#    Deploy policy = best VALIDATION seed per (task, arch), never chosen on test results.
import json, glob, pandas as pd, shutil
rows = []
for p in sorted(glob.glob('hw_logs/training/*/test_summary.json')):
    s = json.load(open(p))
    for c in s['comparisons']:
        rows.append(dict(task=s['task'], arch=s.get('arch', 'attn'), seed=s['seed'], test_set=s.get('test_set'),
                         method=c['method'], policy=s['policy_mean'], baseline=c['mean'],
                         diff=c['policy_minus_method'], win_pct=c['policy_win_pct'], p=c['wilcoxon_p']))
R = pd.DataFrame(rows)
print(R.groupby(['task', 'arch', 'method'])[['policy', 'baseline', 'diff', 'win_pct']].agg(['mean', 'std']).round(3))
print('\ntransformer vs ablation (policy objective, mean +/- SD over seeds; higher is better):')
print(R.drop_duplicates(['task', 'arch', 'seed']).groupby(['task', 'arch']).policy.agg(['mean', 'std', 'count']).round(3))
names = {'attn': 'geo_attn_policy_{}.npz', 'mlp': 'mlp_policy_{}.npz'}
for task in TASKS:
    for arch in ARCHS:
        best, best_val = None, -1e9
        for s in SEEDS:
            c = pd.read_csv(f'hw_logs/training/{task}_{IK}_{arch}_s{s}/curve.csv')
            if c.val_return.max() > best_val: best, best_val = s, c.val_return.max()
        dst = 'deploy/' + names[arch].format(task)
        shutil.copy(f'deploy/policy_{task}_{arch}_s{best}.npz', dst)
        print(f'{task}/{arch}: deploy seed {best} (validation {best_val:.3f}) -> {dst}')

In [ ]:
# 7. Learning curves: transformer (solid) vs ablation (dashed), gap to the best classical planner
import matplotlib.pyplot as plt
fig, axs = plt.subplots(1, len(TASKS), figsize=(10, 3))
for ax, task in zip(axs, TASKS):
    for arch, ls in (('attn', '-'), ('mlp', '--')):
        for s in SEEDS:
            c = pd.read_csv(f'hw_logs/training/{task}_{IK}_{arch}_s{s}/curve.csv')
            ax.plot(c.iter, c.iloc[:, 3], ls, label=f'{arch} s{s}')
    ax.axhline(0, color='k', lw=0.8); ax.set_title(task); ax.set_xlabel('iteration'); ax.set_ylabel('gap to best planner (%)')
    ax.legend(fontsize=6)
plt.tight_layout(); plt.savefig('hw_logs/training/learning_curves.pdf'); plt.show()

In [ ]:
# 8. (after robot evaluation runs) statistics for the paper
import glob
if glob.glob('hw_logs/inspection/*/log.csv'):
    !python analyze_experiments.py "hw_logs/inspection/*/log.csv" --grounding "hw_logs/grounding/validate_*.csv" --out results_hw
else:
    print('no robot evaluation logs yet')

In [ ]:
# 9. Send the trained policies (and refit cost models) back to the Jetson
!python sync.py pack-for-jetson
import glob
files.download(sorted(glob.glob('to_jetson_*.zip'))[-1])